# React to Leptos Converter

Select a model and prompt, paste React code, click **Convert to Leptos**. The generated code appears on the right and is saved as a `.rs` file in `./exports`.


In [1]:
import os
import re
import tempfile
import time
from pathlib import Path

import gradio as gr
import tomllib
from openai import OpenAI

# Use absolute path to the notebook directory
NOTEBOOK_DIR = Path('/home/daniel/Documents/ai-project/AI-AGENTS/react-to-leptos')
MODELS_TOML = NOTEBOOK_DIR / "models.toml"
PROMPTS_DIR = NOTEBOOK_DIR / "prompts"
EXPORT_DIR = NOTEBOOK_DIR / "exports"
os.makedirs(EXPORT_DIR, exist_ok=True)

FENCE_RE = re.compile(r'```(?:\w+)?\s*\n(.*?)```', re.DOTALL)
NEEDS_HUMAN_PREFIX = 'NEEDS-HUMAN'


In [2]:
def load_config(path=MODELS_TOML):
    if not path.exists():
        print(f'Warning: {path} not found. Using defaults.')
        return {"ollama": {"url": "http://localhost:11434/v1"}, "models": {}}
    with open(path, "rb") as f:
        return tomllib.load(f)

CONFIG = load_config()
OLLAMA_URL = CONFIG.get("ollama", {}).get("url", "http://localhost:11434")
# Ensure Ollama URL has /v1 for OpenAI API compatibility
if not OLLAMA_URL.endswith('/v1'):
    OLLAMA_URL = OLLAMA_URL.rstrip('/') + '/v1'
MODELS = CONFIG.get("models", {})

print(f"Models config: {MODELS_TOML}")
print(f"Ollama server: {OLLAMA_URL}")
if MODELS:
    for name, cfg in MODELS.items():
        print(f"  {name} -> {cfg.get('model_id')} (temp={cfg.get('temperature', 0.7)})")
else:
    print("  No models configured in models.toml")

Models config: /home/daniel/Documents/ai-project/AI-AGENTS/react-to-leptos/models.toml
Ollama server: http://localhost:11434/v1
  mistral:latest -> mistral:latest (temp=0.5)
  gemma4:12b -> gemma4:12b (temp=0.6)
  gpt-oss-20b-32k -> gpt-oss-20b-32k:latest (temp=0.6)
  qwen2.5-coder:7b -> qwen2.5-coder:7b (temp=0.2)
  qwen3.8:27b -> qwen3.8:27b (temp=0.5)
  glm-4.7-flash:latest -> glm-4.7-flash:latest (temp=0.7)
  deepseek-v2:16b -> deepseek-v2:16b (temp=0.5)
  devstral-2 -> devstral-2 (temp=0.7)


In [3]:
PROMPT_FILE = re.compile(r"^(?P<name>.+)_v(?P<version>\d+)$")

def load_prompts(directory=PROMPTS_DIR):
    prompts = {}
    if not directory.exists():
        directory.mkdir(parents=True, exist_ok=True)
        print(f'Created prompts directory: {directory}')
    
    for file in sorted(directory.glob("*.md")):
        m = PROMPT_FILE.match(file.stem)
        if not m:
            print(f"Skipping (unexpected name): {file.name}")
            continue
        prompts.setdefault(m["name"], {})[int(m["version"])] = (
            file.read_text(encoding="utf-8").strip()
        )
    return prompts

PROMPTS = load_prompts()
print(f"Prompts directory: {PROMPTS_DIR}")
print(f"Found prompts: {list(PROMPTS.keys())}")
if PROMPTS:
    for name, versions in PROMPTS.items():
        print(name, "->", ", ".join(f"v{v}" for v in sorted(versions)))
else:
    print("No prompts found. Create .md files in prompts/ with format: name_v1.md")

Prompts directory: /home/daniel/Documents/ai-project/AI-AGENTS/react-to-leptos/prompts
Found prompts: ['react_to_leptos']
react_to_leptos -> v1


In [4]:
def strip_fences(text):
    text = (text or '').strip()
    match = FENCE_RE.search(text)
    if match:
        return match.group(1).strip()
    if text.startswith('```'):
        text = '\n'.join(text.splitlines()[1:])
    return text.strip()


def export_leptos(leptos_code):
    if not leptos_code or not leptos_code.strip():
        return None
    fd, path = tempfile.mkstemp(prefix='component-', suffix='.rs', dir=EXPORT_DIR)
    with os.fdopen(fd, 'w', encoding='utf-8') as f:
        f.write(leptos_code.rstrip() + '\n')
    os.chmod(path, 0o644)
    return path


def get_model_cfg(model_name):
    cfg = MODELS.get(model_name, {})
    return {
        'model_id': cfg.get('model_id', model_name),
        'temperature': cfg.get('temperature', 0.0),
        'base_url': OLLAMA_URL,
        'api_key': 'ollama'
    }


def convert(react_code, model_name, prompt_text):
    if not react_code or not react_code.strip():
        return '', None
    
    if not prompt_text or not prompt_text.strip():
        return 'Error: No prompt selected or prompt is empty', None
    
    model_cfg = get_model_cfg(model_name)
    print(f"Connecting to: {model_cfg['base_url']} with model {model_cfg['model_id']}")
    client = OpenAI(base_url=model_cfg['base_url'], api_key=model_cfg['api_key'])
    
    for attempt in range(3):
        try:
            response = client.chat.completions.create(
                model=model_cfg['model_id'],
                temperature=model_cfg['temperature'],
                max_tokens=4096,
                messages=[
                    {'role': 'system', 'content': prompt_text},
                    {'role': 'user', 'content': f'Convert:\n```tsx\n{react_code}\n```'},
                ],
            )
            raw = (response.choices[0].message.content or '').strip()
            
            if raw.startswith(NEEDS_HUMAN_PREFIX):
                return raw, None
            
            code = strip_fences(raw)
            return code, export_leptos(code)
        except Exception as e:
            print(f'Retry {attempt + 1} with {model_cfg["model_id"]}: {type(e).__name__}: {e}')
            time.sleep(2)
    return f'Error: Could not connect to LLM for model {model_cfg["model_id"]}', None


def on_prompt_change(prompt_name):
    if prompt_name and prompt_name in PROMPTS:
        versions = sorted(PROMPTS[prompt_name].keys(), reverse=True)
        text = PROMPTS[prompt_name][versions[0]]
        return text, f"Prompt: {prompt_name} v{versions[0]}"
    return "", "No prompt selected"


In [5]:
with gr.Blocks(title='React to Leptos Converter') as app:
    gr.Markdown('# React to Leptos Converter')
    gr.Markdown('Select model and prompt below, paste React code, click **Convert to Leptos**.')
    
    with gr.Row():
        model_dd = gr.Dropdown(
            choices=list(MODELS.keys()) if MODELS else [],
            value=next(iter(MODELS.keys()), None) if MODELS else None,
            label='Model',
            interactive=bool(MODELS)
        )
        prompt_dd = gr.Dropdown(
            choices=list(PROMPTS.keys()) if PROMPTS else [],
            value=next(iter(PROMPTS.keys()), None) if PROMPTS else None,
            label='Prompt',
            interactive=bool(PROMPTS)
        )
    
    prompt_status = gr.Markdown()
    prompt_text_state = gr.State(value='')
    
    with gr.Row():
        react_input = gr.Textbox(
            label='React Component',
            placeholder='Paste React code here...',
            lines=20,
            max_lines=40,
        )
        with gr.Column():
            leptos_output = gr.Textbox(
                label='Leptos Component',
                placeholder='Leptos code appears here...',
                lines=20,
                max_lines=40,
                interactive=False,
                buttons=['copy'],
            )
            export_file = gr.File(
                label='Download .rs',
                file_count='single',
                interactive=False,
            )
    
    convert_btn = gr.Button('Convert to Leptos', variant='primary')
    clear_btn = gr.ClearButton([react_input, leptos_output, export_file])
    
    prompt_dd.change(
        fn=on_prompt_change,
        inputs=prompt_dd,
        outputs=[prompt_text_state, prompt_status]
    )
    
    convert_btn.click(
        fn=convert,
        inputs=[react_input, model_dd, prompt_text_state],
        outputs=[leptos_output, export_file],
    )
    
    react_input.submit(
        fn=convert,
        inputs=[react_input, model_dd, prompt_text_state],
        outputs=[leptos_output, export_file],
    )
    
    app.load(
        fn=lambda: on_prompt_change(next(iter(PROMPTS.keys()), None) if PROMPTS else None),
        outputs=[prompt_text_state, prompt_status]
    )

In [6]:
app.launch()


* Running on local URL:  http://127.0.0.1:7862
* To create a public link, set `share=True` in `launch()`.


Connecting to: http://localhost:11434/v1 with model mistral:latest
